# Notebook 02: The Agent Loop

**Learning objectives:**
- Understand what makes a system 'agentic' vs. a standard ML pipeline
- See how tool calling works at the API level (no framework)
- Read the agent's reasoning trace and understand each decision
- Observe how the agent adapts to a different scene

**Prerequisites:** Run Notebook 00 and 01 first (or at least 00).

In [1]:
import sys, os
from pathlib import Path
# Move to project root so relative data paths resolve correctly
_here = Path(os.getcwd())
if not (_here / 'data' / 'scenes').exists() and (_here.parent / 'data' / 'scenes').exists():
    os.chdir(_here.parent)
sys.path.insert(0, str(Path(os.getcwd())))

import inspect
from agent.scene import load_scene
from agent.types import BoundingBox
from agent import tools
from agent.schemas import TOOL_SCHEMAS
from agent.loop import run_agent, SYSTEM_PROMPT
from agent.mock import run_mock_agent

load_scene('scene_a')
print('Scene A loaded. Ready.')

Scene A loaded. Ready.


## Step 1: One Tool Example

Before the agent, let's confirm: tools are just functions.

In [2]:
bb = BoundingBox(0, 200, 0, 200)
result = tools.compute_ndvi(bb)
print(result.summary)

NDVI for rows 0–200, cols 0–200: mean=0.650, std=0.225. Pixels below 0.3 (stressed): 14.1%. ⚠️ Significant stressed area detected.


## Step 2: One Schema Example

This is how the model learns what `compute_ndvi` accepts:

In [3]:
import json
ndvi_schema = next(s for s in TOOL_SCHEMAS if s['name'] == 'compute_ndvi')
print(json.dumps(ndvi_schema, indent=2))

{
  "name": "compute_ndvi",
  "description": "Compute NDVI (Normalized Difference Vegetation Index) for a region. NDVI = (NIR - Red) / (NIR + Red). Values near 1 = dense healthy vegetation; below 0.3 = stress, sparse vegetation, or bare soil. Returns mean, std, stressed-pixel fraction, and an image path.",
  "input_schema": {
    "type": "object",
    "properties": {
      "region": {
        "type": "object",
        "description": "Bounding box in pixel coordinates (integers).",
        "properties": {
          "row_min": {
            "type": "integer",
            "description": "Start row (inclusive)"
          },
          "row_max": {
            "type": "integer",
            "description": "End row (exclusive)"
          },
          "col_min": {
            "type": "integer",
            "description": "Start column (inclusive)"
          },
          "col_max": {
            "type": "integer",
            "description": "End column (exclusive)"
          }
        },
      

## Step 3: The System Prompt

Read this carefully — every instruction shapes agent behavior.

Note especially:
- *"Explain your reasoning before each tool call"* → creates the trace we learn from
- *"Only call tools when warranted"* → prevents exhaustive tool use
- *"Express uncertainty when appropriate"* → prevents overconfident reports

In [4]:
print(SYSTEM_PROMPT)

You are an autonomous crop health analyst working with satellite imagery. Your job is to assess crop health in a given region and produce a diagnostic report.

You have access to tools that compute spectral indices (NDVI, NDWI, EVI), retrieve timeseries data, flag anomalous regions, and compare current conditions to historical baselines.

**Your workflow**:
1. Start with a broad assessment (e.g., compute NDVI for the region)
2. If you detect anomalies or low values, investigate further using additional indices or timeseries
3. Explain your reasoning in plain text before each tool call. Describe what you observed and why you're choosing the next tool.
4. Only call tools when warranted by your observations. You do not need to use all available tools.
5. If tools return errors, adapt your strategy and explain what happened.
6. When you have sufficient information, produce a final diagnostic report with:
   - Summary of findings
   - Specific locations of concern (if any)
   - Likely cause

## Step 4: The Loop

The entire agent loop is ~50 lines. Read it:

In [5]:
from agent.loop import run_agent
print(inspect.getsource(run_agent))

def run_agent(
    user_request: str,
    client: Optional[anthropic.Anthropic] = None,
    temperature: float = 0,
    max_iterations: int = 20,
    verbose: bool = True,
) -> str:
    """Run the agent loop. Returns the final text response."""
    if client is None:
        client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    messages = [{"role": "user", "content": user_request}]
    final_text = ""

    for _ in range(max_iterations):
        response = client.messages.create(
            model=MODEL,
            system=SYSTEM_PROMPT,
            tools=TOOL_SCHEMAS,
            cache_control={"type": "ephemeral"},
            messages=messages,
            temperature=temperature,
            max_tokens=4096,
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += bl

## Step 5: Scripted Run — Scene A

We use **mock mode** here so the trace is identical every time. The model responses are pre-recorded, but **all tool calls are real** (live numpy computation).

Watch each `[TOOL CALL]` block:
1. What did the agent observe that prompted this call?
2. What new information did it learn?
3. What might it do next?

In [6]:
load_scene('scene_a')
run_mock_agent('scene_a')


[MOCK MODE] Replaying pre-recorded trace for scene_a

[AGENT REASONING]
I'll begin with a broad NDVI assessment to understand the overall vegetation health before looking at specific anomalies.

[TOOL CALL] compute_ndvi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 200,
    "col_min": 0,
    "col_max": 200
  }
}
[RESULT] NDVI for rows 0–200, cols 0–200: mean=0.650, std=0.225. Pixels below 0.3 (stressed): 14.1%. ⚠️ Significant stressed area detected.

[AGENT REASONING]
The NDVI shows a significant stressed area (>25% of pixels below 0.3). I'll flag the anomalous regions to get precise bounding boxes.

[TOOL CALL] flag_anomalous_regions
[INPUT] {
  "index": "ndvi",
  "threshold": 0.4,
  "direction": "below"
}
[RESULT] Found 4 anomalous region(s) where NDVI is below 0.40.
Total anomalous pixels: 8605 (21.5% of scene).
Region 1: rows 0–50, cols 0–50, 2128 px, mean NDVI=0.254.
Region 2: rows 0–50, cols 50–100, 2180 px, mean NDVI=0.250.
Region 3: rows 50–100, cols 0–50, 2146 px, 

'## Crop Health Diagnostic Report\n\n**Summary of findings:**\nSignificant vegetation stress in the northwest quadrant (rows 0–100, cols 0–100), affecting ~25% of the total scene area.\n\n**Specific locations of concern:**\n- Northwest quadrant: mean NDVI dropped from ~0.65 (baseline) to ~0.25 (current), a Δ ≈ −0.40.\n\n**Likely cause: Water stress (HIGH confidence)**\n- NDWI is strongly negative (< −0.2) in the affected area — reduced canopy water content.\n- Timeseries shows a sharp NDVI decline over the past 2–3 weeks, consistent with irrigation failure or drought onset, not seasonal phenology.\n- Baseline comparison confirms the NW quadrant was healthy 30 days ago.\n\n**Recommended follow-up actions:**\n1. Inspect irrigation infrastructure in the northwest quadrant immediately.\n2. Check field water meters and irrigation schedules for the past month.\n3. Consider emergency irrigation if crops are at a critical growth stage.\n4. Re-assess in 7–10 days after intervention to monitor r

### Discussion Questions

1. After `compute_ndvi`: Why did the agent choose `flag_anomalous_regions` next rather than immediately calling `compute_ndwi`?
2. After `flag_anomalous_regions`: The agent now calls `compute_ndwi` on the *anomalous sub-region*, not the full scene. Why is this more efficient?
3. After `compute_ndwi`: The NDWI is negative. Could this mean water stress, bare soil, or something else? What rules this out?
4. After `get_pixel_timeseries`: Why does recent onset matter for diagnosis?
5. The agent called `compare_to_baseline` last, not first. Was this the right order?

## Step 6: Live Run — Scene B

Scene B is ambiguous: mild uniform NDVI depression, no strong spatial anomaly, no baseline.

**Watch:** Does the agent take a different path than it did for Scene A?

Warning: **Requires `ANTHROPIC_API_KEY`** in your environment. Estimated cost: ~$0.10–0.20 for this run.

If you don't have an API key, skip this cell — the take-home exercises cover this.

In [7]:
from dotenv import load_dotenv

load_dotenv()  # load our API key into our environment

# LIVE API CALL
if os.environ.get('ANTHROPIC_API_KEY'):
    load_scene('scene_b')
    run_agent(
        'Assess crop health for the entire scene. '
        'Identify any areas of concern and determine likely causes.',
        temperature=0,
    )
else:
    print('ANTHROPIC_API_KEY not set — skipping live run.')
    print('Set your key and re-run this cell to see the live agent in action.')


[TOOL CALL] compute_ndvi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 512,
    "col_min": 0,
    "col_max": 512
  }
}
[RESULT] NDVI for rows 0–512, cols 0–512: mean=0.556, std=0.057. Pixels below 0.3 (stressed): 0.0%. Vegetation appears generally healthy.

[TOOL CALL] flag_anomalous_regions
[INPUT] {
  "index": "ndvi",
  "threshold": 0.4,
  "direction": "below"
}
[RESULT] No anomalous pixels found: NDVI below 0.40.

[TOOL CALL] compute_ndwi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 512,
    "col_min": 0,
    "col_max": 512
  }
}
[RESULT] NDWI for rows 0–512, cols 0–512: mean=-0.412, std=0.100. Status: water-stressed (dry canopy). Negative-NDWI pixels: 100.0%.

[TOOL CALL] compute_evi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 512,
    "col_min": 0,
    "col_max": 512
  }
}
[RESULT] EVI for rows 0–512, cols 0–512: mean=0.424, std=0.071. Pixels below 0.2: 0.0%. EVI is less saturated than NDVI in dense canopy — use to verify NDVI findings.

[TOOL 

/home/ryan/projects/agentic-satellite-data/agent/tools.py:94: RuntimeWarning: Mean of empty slice
  mean_val = float(ndwi.mean())
/home/ryan/projects/agentic-satellite-data/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/home/ryan/projects/agentic-satellite-data/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ryan/projects/agentic-satellite-data/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/ryan/projects/agentic-satellite-data/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/home/ryan/projects/agentic-satellite-data/agent

[RESULT] NDWI for rows 256–512, cols 0–256: mean=nan, std=nan. Status: moderate moisture. Negative-NDWI pixels: nan%.

[TOOL CALL] compute_ndwi
[INPUT] {
  "region": {
    "row_min": 256,
    "row_max": 512,
    "col_min": 256,
    "col_max": 512
  }
}
[RESULT] NDWI for rows 256–512, cols 256–512: mean=nan, std=nan. Status: moderate moisture. Negative-NDWI pixels: nan%.

[TOOL CALL] compute_ndvi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 256,
    "col_min": 0,
    "col_max": 256
  }
}
[RESULT] NDVI for rows 0–256, cols 0–256: mean=0.556, std=0.057. Pixels below 0.3 (stressed): 0.0%. Vegetation appears generally healthy.

[TOOL CALL] compute_evi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 256,
    "col_min": 0,
    "col_max": 256
  }
}
[RESULT] EVI for rows 0–256, cols 0–256: mean=0.424, std=0.071. Pixels below 0.2: 0.0%. EVI is less saturated than NDVI in dense canopy — use to verify NDVI findings.

[TOOL CALL] get_pixel_timeseries
[INPUT] {
  "lat": 128,
  "